In [1]:
import boto3

bucket = "rimas-olist-clv-project2"

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="ml/churn/"
)

for item in response.get("Contents", []):
    print(item["Key"])

ml/churn/
ml/churn/current-snapshot/
ml/churn/current-snapshot/olist_customer_features_current_snapshot.csv


In [2]:
import pandas as pd
import io
import boto3

bucket = "rimas-olist-clv-project2"
key = "ml/churn/current-snapshot/olist_customer_features_current_snapshot.csv"

s3 = boto3.client("s3")

obj = s3.get_object(
    Bucket=bucket,
    Key=key
)

df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print("Shape:", df.shape)

df.head()

Shape: (93358, 7)


,customer_unique_id,recency,frequency,monetary,avg_order_value,churned,customer_segment
0,b577af9a54b023b4bcc51f69bb937f1c,199,1,170.43,170.43,1,Low Value
1,2c613e1daa937ba2dda45140c410f48f,433,1,77.03,77.03,1,Low Value
2,9949e007c1f037474a3423372cccb701,441,1,104.37,104.37,1,Low Value
3,0bd4d2600b49eb951489ebc82723a4e6,70,1,112.77,112.77,0,Low Value
4,50343a5529a9ab46eacfd1fdc387cf0a,231,1,64.79,64.79,1,Low Value


In [3]:
df.columns.tolist()

['customer_unique_id',
 'recency',
 'frequency',
 'monetary',
 'avg_order_value',
 'churned',
 'customer_segment']

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93358 entries, 0 to 93357
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_unique_id  93358 non-null  object 
 1   recency             93358 non-null  int64  
 2   frequency           93358 non-null  int64  
 3   monetary            93357 non-null  float64
 4   avg_order_value     93357 non-null  float64
 5   churned             93358 non-null  int64  
 6   customer_segment    93358 non-null  object 
dtypes: float64(2), int64(3), object(2)
memory usage: 5.0+ MB


In [5]:
df["churned"].value_counts()

churned
1    55799
0    37559
Name: count, dtype: int64

In [6]:
df["churned"].value_counts(normalize=True) * 100

churned
1    59.768847
0    40.231153
Name: proportion, dtype: float64

In [7]:
df["customer_segment"].value_counts()

customer_segment
Low Value       73371
Medium Value    15722
High Value       4265
Name: count, dtype: int64

In [9]:
!pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [21]:
from sklearn.model_selection import train_test_split

X = df[
    [
        "recency",
        "frequency",
        "monetary",
        "avg_order_value"
    ]
]

y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 74685
Testing rows: 18672


In [22]:
print(X.isnull().sum())

recency            0
frequency          0
monetary           0
avg_order_value    0
dtype: int64


In [23]:
print(X_train.isnull().sum())

recency            0
frequency          0
monetary           0
avg_order_value    0
dtype: int64


In [24]:
print(X_train.describe())

            recency     frequency      monetary  avg_order_value
count  74685.000000  74685.000000  74685.000000     74685.000000
mean     240.569298      1.034036    165.159353       160.218098
std      152.506444      0.211332    225.766811       218.934260
min        3.000000      1.000000      9.590000         9.590000
25%      117.000000      1.000000     63.100000        62.390000
50%      221.000000      1.000000    108.000000       105.940000
75%      349.000000      1.000000    182.360000       176.260000
max      698.000000     15.000000  13664.080000     13664.080000


In [25]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
ROC AUC  : 1.0


In [27]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[ 7512     0]
 [    0 11160]]


In [28]:
feature_importance = pd.DataFrame(
    {
        "Feature": X.columns,
        "Coefficient": model.coef_[0]
    }
)

feature_importance.sort_values(
    by="Coefficient",
    ascending=False
)


,Feature,Coefficient
0,recency,6.624151
3,avg_order_value,0.001872
2,monetary,-0.001999
1,frequency,-0.048910


In [41]:
import pandas as pd
import boto3
import io

bucket = "rimas-olist-clv-project2"
key = "ml/clv/olist_clv_training_dataset.csv"

s3 = boto3.client("s3")

obj = s3.get_object(
    Bucket=bucket,
    Key=key
)

clv_df = pd.read_csv(
    io.BytesIO(obj["Body"].read())
)

print(clv_df.shape)

clv_df.head()

(57509, 6)


,customer_unique_id,recency,frequency,monetary,avg_order_value,future_spend
0,9f10ed5e6157a32cce16e0b22013096b,239,1,71.14,71.140,0.00
1,7a97eb62430c9e27f4a13220d267b9af,274,1,145.65,145.650,0.00
2,a1f6aec75e41168caba548372f245c68,71,1,122.50,122.500,125.63
3,02bb821f9cd6c3079ecb9fed96232e18,105,1,96.81,96.810,0.00
4,be31ab006786747df81854035a1960f4,58,2,100.07,50.035,0.00


In [42]:
clv_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57509 entries, 0 to 57508
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_unique_id  57509 non-null  object 
 1   recency             57509 non-null  int64  
 2   frequency           57509 non-null  int64  
 3   monetary            57509 non-null  float64
 4   avg_order_value     57509 non-null  float64
 5   future_spend        57509 non-null  float64
dtypes: float64(3), int64(2), object(1)
memory usage: 2.6+ MB


In [43]:
clv_df.isnull().sum()

customer_unique_id    0
recency               0
frequency             0
monetary              0
avg_order_value       0
future_spend          0
dtype: int64

In [44]:
customer_ids = clv_df["customer_unique_id"]

X = clv_df[
    [
        "recency",
        "frequency",
        "monetary",
        "avg_order_value"
    ]
]

y = clv_df["future_spend"]

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y,
    customer_ids,
    test_size=0.2,
    random_state=42
)

In [46]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print("CLV model trained.")

CLV model trained.


In [47]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

predictions = model.predict(X_test)

results = pd.DataFrame({
    "customer_unique_id": id_test,
    "actual_future_spend": y_test,
    "predicted_clv": predictions
})

all_predictions = model.predict(X)

clv_scores = pd.DataFrame({
    "customer_unique_id": clv_df["customer_unique_id"],
    "predicted_clv": all_predictions
})

mae = mean_absolute_error(y_test, predictions)

mse = mean_squared_error(
    y_test,
    predictions
)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, predictions)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 3.558161269947727
RMSE: 19.83938881228442
R²  : 0.00015486593176028407


In [48]:
# results = X_test.copy()

# results["actual_future_spend"] = y_test
# results["predicted_future_spend"] = predictions

In [49]:
results.to_csv(
    "clv_test_predictions.csv",
    index=False
)

clv_scores.to_csv(
    "clv_predictions.csv",
    index=False
)

In [50]:
metrics = pd.DataFrame(
    {
        "model": ["Linear Regression"],
        "MAE": [mae],
        "RMSE": [rmse],
        "R2": [r2]
    }
)

metrics

,model,MAE,RMSE,R2
0,Linear Regression,3.558161,19.839389,0.000155


In [51]:
metrics.to_csv(
    "clv_model_metrics.csv",
    index=False
)